In [ ]:
import sys
from pathlib import Path

repo_name = 'triplet-phase-lock'
repo_url = 'https://github.com/thinkthoughts/triplet-phase-lock.git'
repo_root = Path('/content') / repo_name

if not repo_root.exists():
    !git clone {repo_url}

if not (repo_root / 'src').exists():
    raise FileNotFoundError(f"Expected src/ at {repo_root / 'src'}, but it was not found.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root added to path:', repo_root)
print('src exists:', (repo_root / 'src').exists())


# 03 — what resists?

**triplet-phase-lock**

Π stage: what resists?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.expand import sequence_n, sequence_n_perturbed, sequence_n_noisy, build_triplets_from_values
from src.extend import local_delta
from src.resist import cosine_scores, empirical_clean_reference, strict_mask, threshold_45, threshold_strict_default
from src.metrics import resistance_ratio

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True

In [ ]:
num_steps = 80
k = np.arange(1, num_steps + 3)

clean_vals = sequence_n(k)
weak_vals = sequence_n_perturbed(k, amplitude=8.0, period=6.0)
strong_vals = sequence_n_perturbed(k, amplitude=20.0, period=4.0)
noisy_vals = sequence_n_noisy(k, amplitude=40.0, seed=7)

triplets_clean = build_triplets_from_values(clean_vals)
triplets_weak = build_triplets_from_values(weak_vals)
triplets_strong = build_triplets_from_values(strong_vals)
triplets_noisy = build_triplets_from_values(noisy_vals)

In [ ]:
reference_global = np.array([1.0, 1.0, 1.0])
reference_clean = empirical_clean_reference(triplets_clean)

thr45 = threshold_45()
thrS = threshold_strict_default()

print(f'45° threshold    : {thr45:.12f}')
print(f'Strict threshold : {thrS:.12f}')

In [ ]:
scores_global = {
    'clean': cosine_scores(triplets_clean, reference_global),
    'weak': cosine_scores(triplets_weak, reference_global),
    'strong': cosine_scores(triplets_strong, reference_global),
    'noisy': cosine_scores(triplets_noisy, reference_global),
}

score_idx = np.arange(1, len(scores_global['clean']) + 1)

plt.figure()
for n in ['clean', 'weak', 'strong', 'noisy']:
    plt.plot(score_idx, scores_global[n], marker='o', label=n)
plt.axhline(thr45, linestyle='--', label='45° threshold')
plt.xlabel('k')
plt.ylabel(r'$\cos\theta_k$')
plt.title('Global-reference resistance scores')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
scores = {
    'clean': cosine_scores(triplets_clean, reference_clean),
    'weak': cosine_scores(triplets_weak, reference_clean),
    'strong': cosine_scores(triplets_strong, reference_clean),
    'noisy': cosine_scores(triplets_noisy, reference_clean),
}

plt.figure()
for n in ['clean', 'weak', 'strong', 'noisy']:
    plt.plot(score_idx, scores[n], marker='o', label=n)
plt.axhline(thr45, linestyle='--', label='45° threshold')
plt.axhline(thrS, linestyle='--', label='strict threshold')
plt.xlabel('k')
plt.ylabel(r'$\cos\theta_k$')
plt.title('Clean-reference resistance scores')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
mask45 = {n: scores[n] >= thr45 for n in scores}
maskS = {n: strict_mask(scores[n], thrS) for n in scores}

print(f"45° ratios   : {[f'{n}={resistance_ratio(mask45[n]):.6f}' for n in ['clean','weak','strong','noisy']]}")
print(f"Strict ratios: {[f'{n}={resistance_ratio(maskS[n]):.6f}' for n in ['clean','weak','strong','noisy']]}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

for n in ['clean', 'weak', 'strong', 'noisy']:
    axes[0].plot(score_idx, mask45[n].astype(float), marker='o', label=n)
axes[0].set_ylabel('accepted / rejected')
axes[0].set_title('Resistance masks under the 45° threshold')
axes[0].legend()

for n in ['clean', 'weak', 'strong', 'noisy']:
    axes[1].plot(score_idx, maskS[n].astype(float), marker='o', label=n)
axes[1].set_xlabel('k')
axes[1].set_ylabel('accepted / rejected')
axes[1].set_title('Resistance masks under the strict threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
accepted_delta = {
    'clean': local_delta(triplets_clean[maskS['clean']]),
    'weak': local_delta(triplets_weak[maskS['weak']]),
    'strong': local_delta(triplets_strong[maskS['strong']]),
    'noisy': local_delta(triplets_noisy[maskS['noisy']]),
}

plt.figure()
for n in ['clean', 'weak', 'strong', 'noisy']:
    if len(accepted_delta[n]):
        plt.plot(np.arange(1, len(accepted_delta[n]) + 1), accepted_delta[n], marker='o', label=f'{n} accepted Δ')
plt.xlabel('accepted-step index')
plt.ylabel(r'$\Delta$ among accepted triplets')
plt.title('Local variation after strict resistance filtering')
plt.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook establishes the **resist** stage:

- compare global and empirical clean references
- compare broad and strict thresholds
- show threshold-dependent acceptance